# 10 -- Ambient periods, HVSR and amplification

Object-driven microtremor analysis: condition the object, then `ds.ambient(...)` computes everything and the plots only draw.

In [ ]:
%matplotlib inline

from datetime import datetime
from asdea_sensors import SensorDataset
from asdea_sensors.plotting import ambient_plots

DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"
time_start = datetime(2026, 5, 31, 15, 0, 0)

In [ ]:
ds = SensorDataset(DATA, parallel=False, verbose=True)
ds.devices

## Conditioned object for ambient

For microtremor periods use **acceleration without a band-pass** (a filter would shape the spectrum). Window + baseline is enough.

In [ ]:
ds_amb = ds.window(start=time_start, length="10min").baseline()

## Ambient analysis (object-driven)

Pass only the analysis parameters; `Fs` comes from the object. Returns `{device: result}`.

In [ ]:
amb = ds_amb.ambient(
    sta       = 1.0,
    lta       = 30.0,
    vent      = 30.0,
    vmin      = 0.7,
    vmax      = 1.4,
    p         = 0.05,
    bexp      = 80,
    component = "x",
    kind      = "acc",   # "acc" | "vel" | "disp" (vel/disp need .derive())
)
print({d: round(amb[d]["f_dom"], 3) for d in amb})   # dominant frequency [Hz]

## STA/LTA, selected windows and spectra

In [ ]:
# STA/LTA ratio + vmin/vmax band, all sensors
ambient_plots.plot_sta_lta_all(amb, layout="auto", ylim=(0, 3))

In [ ]:
# Selected time windows (rainbow) over each signal
ambient_plots.plot_windows_all(amb, layout="auto")

In [ ]:
# Per-sensor spectra: all windows (gray) + mean (color) + peaks
ambient_plots.plot_spectrum_all(amb, peak_spacing_hz=0.2, num_peaks=4, min_freq=0.3)

In [ ]:
# Mean spectra compared across sensors
dom = ambient_plots.plot_mean_spectrum_all(amb, component="x", layout="overlay", xlim=(0, 25))
dom

## HVSR (Nakamura), per sensor

In [ ]:
hv = ds_amb.MOF00135.hvsr(
    sta=1.0, lta=30.0, vent=30.0, vmin=0.7, vmax=1.4, p=0.05, bexp=80,
    combine="squared", comp_h=("x", "y"), comp_v="z", f0_min=0.2, f0_max=20.0,
)
print("HVSR f0 =", round(float(hv["f0"]), 2), "Hz")

## Spectral amplification (floor vs base)

In [ ]:
amp = ds_amb.amplification(
    ref       = "MOF00134",
    others    = ["MNAT0031", "MOF00135", "MOF00136"],
    basis     = "fourier",   # "fourier" | "response" | "hvsr"
    component = "x",
    config    = {"Fs": ds.fs, "STA": 1.0, "LTA": 30.0, "vent": 30.0,
               "vmin": 0.7, "vmax": 1.4, "p": 0.05, "bexp": 80},
)
print("basis:", amp["basis"], " devices:", list(amp["ratios"].keys()))